In [4]:
# Cell 1: config & imports
import os, time, json, math, random
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sentence_transformers import SentenceTransformer
from umap import UMAP
import hdbscan
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics.pairwise import cosine_similarity

In [29]:
# reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

# Paths & params (edit if needed)
INPUT_CSV = "/Users/suyashmali/Case_Study_Materials/LLM/Notebook/Cleaned_Data_New.csv"  
OUT_DIR = "./taxonomy_experiments"
os.makedirs(OUT_DIR, exist_ok=True)

TEXT_COL = "metadata_clean" 
LABEL_COL = "Level3"          
K_PER_CATEGORY = 100          

EMBED_MODEL = "all-MiniLM-L6-v2"
USE_UMAP = True
UMAP_N_COMPONENTS = 25
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.0

HDB_MIN_CLUSTER_SIZE = 10

In [30]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option("max_colwidth", None)

In [8]:
# # Cell 2: load and balanced sample
# df = pd.read_csv("/Users/suyashmali/Case_Study_Materials/LLM/Notebook/Cleaned_Data_New.csv")
# print("Rows in original df:", len(df))

# label_col = "Level3"   
# text_col = "metadata_clean"       

# # basic check
# if label_col not in df.columns:
#     raise ValueError(f"{label_col} not found in dataframe. Choose the correct label column name.")

# # drop rows with missing label (for balanced sampling only)
# df_labelled = df[~df[label_col].isna() & df[label_col].astype(str).str.strip().ne("")].copy()
# print("Labelled rows:", len(df_labelled))

# def build_balanced_sample(df, label_col, k=100, random_state=RANDOM_STATE):
#     groups = []
#     for label, g in df.groupby(label_col):
#         n = min(len(g), k)
#         sampled = g.sample(n=n, random_state=random_state)
#         groups.append(sampled)
#     sample = pd.concat(groups).reset_index(drop=True)
#     return sample

# sample_df = build_balanced_sample(df_labelled, label_col, k=K_PER_CATEGORY)
# print("Balanced sample size:", len(sample_df))
# print("Unique labels in sample:", sample_df[label_col].nunique())
# sample_df.head()

Rows in original df: 289865
Labelled rows: 289865
Balanced sample size: 16347
Unique labels in sample: 209


,Brand,BrandPartCode,ProductName,Description.LongProductName,Description.LongDesc,SummaryDescription.LongSummaryDescription,pathlist_names,Level1,Level2,Level3,metadata_text,metadata_clean
0,Vaddio,999-9555-001,999-9555-001,"HDMI In, HDMI Out, IR, LAN, RJ-45, DC, EU/UK",Vaddio’s OneLINK System for Cisco® C-Series Co...,Vaddio 999-9555-001. Product colour: Black. Po...,Computers & Electronics>Networking>AV Conferen...,Computers & Electronics,Networking,AV Conferencing Bridges,Vaddio 999-9555-001 AV conferencing bridge Eth...,vaddio 999-9555-001 av conferencing bridge eth...
1,StarTech.com,MOD4AVHD,Audio / Video Module for Conference Table Conn...,StarTech.com Audio / Video Module for Conferen...,Boardroom and Huddle Space Connectivity | 4K |...,StarTech.com Audio / Video Module for Conferen...,Computers & Electronics>Networking>AV Conferen...,Computers & Electronics,Networking,AV Conferencing Bridges,StarTech.com Audio / Video Module for Conferen...,startech.com audio / video module for conferen...
2,Vaddio,999-9540-001,999-9540-001,"HDMI In, HDMI Out, 2 x RS-232, RJ-45, DC, EU/UK",<b>More Flexibility for Room Design and Instal...,Vaddio 999-9540-001. Product colour: Black. Ca...,Computers & Electronics>Networking>AV Conferen...,Computers & Electronics,Networking,AV Conferencing Bridges,Vaddio 999-9540-001 AV conferencing bridge Eth...,vaddio 999-9540-001 av conferencing bridge eth...
3,Vaddio,999-8530-001,EasyUSB Mixer/Amp,"20 - 20000 HZ, >90dB, THD + Noise, RCA, 2- Pin...",- Supports up to two EasyMic Devices (MicPOD o...,Vaddio EasyUSB Mixer/Amp. Frequency range: 20 ...,Computers & Electronics>Networking>AV Conferen...,Computers & Electronics,Networking,AV Conferencing Bridges,Vaddio EasyUSB Mixer/Amp | Vaddio | EasyUSB Mi...,vaddio easyusb mixer/amp vaddio easyusb mixer/...
4,Logitech,960-001094,960-001094,"SmartDock + Extender Box, 1080p, HDMI, USB, LAN",<b>SmartDock Makes Skype Rooms Easy</b><br>\nA...,Logitech 960-001094. Maximum resolution: 1920 ...,Computers & Electronics>Networking>AV Conferen...,Computers & Electronics,Networking,AV Conferencing Bridges,Logitech 960-001094 AV conferencing bridge 192...,logitech 960-001094 av conferencing bridge 192...


In [33]:
# Cell 1: Load data and build balanced sample (if you already have sample_df, skip)
df = pd.read_csv(INPUT_CSV)
print("Original rows:", len(df))

# If sample_df exists from earlier, you can skip the sampling step and do:
# sample_df = df_sampled_loaded
# Otherwise do balanced sampling:
df_labelled = df[~df[LABEL_COL].isna() & df[LABEL_COL].astype(str).str.strip().ne("")].copy()
print("Labelled rows:", len(df_labelled))

def build_balanced_sample(df, label_col, k=100, random_state=RANDOM_STATE):
    groups = []
    for label, g in df.groupby(label_col):
        n = min(len(g), k)
        sampled = g.sample(n=n, random_state=random_state)
        groups.append(sampled)
    sample = pd.concat(groups).reset_index(drop=True)
    return sample

sample_df = build_balanced_sample(df_labelled, LABEL_COL, k=K_PER_CATEGORY)
print("Balanced sample size:", len(sample_df))
print("Unique labels in sample:", sample_df[LABEL_COL].nunique())

# Save sample for reproducibility:
# sample_df.to_csv(os.path.join(OUT_DIR, "balanced_sample.csv"), index=False)
sample_df.head()


Original rows: 289865
Labelled rows: 289865
Balanced sample size: 16347
Unique labels in sample: 209


,Brand,BrandPartCode,ProductName,Description.LongProductName,Description.LongDesc,SummaryDescription.LongSummaryDescription,pathlist_names,Level1,Level2,Level3,metadata_text,metadata_clean
0,Vaddio,999-9555-001,999-9555-001,"HDMI In, HDMI Out, IR, LAN, RJ-45, DC, EU/UK","Vaddio’s OneLINK System for Cisco® C-Series Codecs (C20, C40, C60 and C90) and Cisco® Precision HD (1080p 12x and 720p) cameras is digital camera extension system that extends HDMI video, power and network control on a single Cat-5e or better cable at distances of up to 328’ (100m). OneLINK sends and regulates power to the camera, provides a bi-directional control channel for Ethernet, and transmits/receives uncompressed HDMI digital video up to 1080p/60. Vaddio OneLINK - the art of easy - making installations and integration simple.","Vaddio 999-9555-001. Product colour: Black. Power source type: DC. Cables included: AC,HDMI,LAN (RJ-45)",Computers & Electronics>Networking>AV Conferencing Bridges,Computers & Electronics,Networking,AV Conferencing Bridges,"Vaddio 999-9555-001 AV conferencing bridge Ethernet LAN Black | Vaddio | 999-9555-001 | AV Conferencing Bridges | Vaddio 999-9555-001. Product colour: Black. Power source type: DC. Cables included: AC,HDMI,LAN (RJ-45) | Vaddio 999-9555-001, Black, DC, AC,HDMI,LAN (RJ-45) | HDMI In, HDMI Out, IR, LAN, RJ-45, DC, EU/UK | Vaddio’s OneLINK System for Cisco® C-Series Codecs (C20, C40, C60 and C90) and Cisco® Precision HD (1080p 12x and 720p) cameras is digital camera extension system that extends HDMI video, power and network control on a single Cat-5e or better cable at distances of up to 328’ (100m). OneLINK sends and regulates power to the camera, provides a bi-directional control channel for Ethernet, and transmits/receives uncompressed HDMI digital video up to 1080p/60. Vaddio OneLINK - the art of easy - making installations and integration simple.","vaddio 999-9555-001 av conferencing bridge ethernet lan black vaddio 999-9555-001 av conferencing bridges vaddio 999-9555-001. product colour black. power source type dc. cables included ac,hdmi,lan rj-45 vaddio 999-9555-001, black, dc, ac,hdmi,lan rj-45 hdmi in, hdmi out, ir, lan, rj-45, dc, eu/uk vaddio s onelink system for cisco c-series codecs c20, c40, c60 and c90 and cisco precision hd 1080p 12x and 720p cameras is digital camera extension system that extends hdmi video, power and network control on a single cat-5e or better cable at distances of up to 328 100m . onelink sends and regulates power to the camera, provides a bi-directional control channel for ethernet, and transmits/receives uncompressed hdmi digital video up to 1080p/60. vaddio onelink - the art of easy - making installations and integration simple."
1,StarTech.com,MOD4AVHD,Audio / Video Module for Conference Table Connectivity Box,"StarTech.com Audio / Video Module for Conference Table Connectivity Box - 4K - HDMI, DP, VGA - Table-Mounting Bracket Included","Boardroom and Huddle Space Connectivity | 4K | HDMI, DP, VGA | Table-Mounting Bracket Included<br/><br/>Add convenient A/V connectivity and network access to your boardroom, classroom or huddle space. This audio/video module makes it easy to connect HDMI, DisplayPort and VGA laptops to your room’s HDMI display. <br/><br/>Conveniently Connect to a Display <br/><br/>Modern office environments need to support technology more than ever, to keep laptops, phones and tablets connected. During meetings, lectures and conferences, you may need to present content from your laptop to a display or projector. <br/><br/>This A/V module simplifies connectivity and collaboration with automatic switching and conversion. It lets you connect your HDMI, DP or VGA laptop to a shared HDMI display or projector, without the hassle of menu and input selections. And, the module automatically switches to the most recently connected or powered-on laptop. Installing the module is easy, requiring just a single HDMI cable between the module and the display. <br/><br/>C

In [26]:
sample_df.columns

Index(['Brand', 'BrandPartCode', 'ProductName', 'Description.LongProductName',
       'Description.LongDesc', 'SummaryDescription.LongSummaryDescription',
       'pathlist_names', 'Level1', 'Level2', 'Level3', 'metadata_text',
       'metadata_clean'],
      dtype='object')

In [13]:
sample_df.tail(5)

,Brand,BrandPartCode,ProductName,Description.LongProductName,Description.LongDesc,SummaryDescription.LongSummaryDescription,pathlist_names,Level1,Level2,Level3,metadata_text,metadata_clean
16342,3M,FT600003279,Gel Wrist Rest Platform,Gel Wrist Rest Platform,"Sleek in appearance and in operation, this platform is designed with tapered wrist rests and rounded corners for comfort beyond compare. Use the tilt feature to help keep wrists properly aligned and<br>\r\nimprove ergonomic posture.<br>\r\n<br>\r\n- Keyboard and mouse wrist rest with antimicrobial product protection adds comfort by minimizing pressure points<br>\r\n- Battery saving design on mouse surfaces extends battery life of wireless mice up to 75%†<br>\r\n- Leatherette surfaces on wrist rests are buttery-soft and easy to keep clean<br>\r\n<br>\r\n- Tilt-adjustable platform for keyboard and mouse<br>\r\n- Mousing surface is repositionable for left-handed mousing","3M Gel Wrist Rest Platform. Product colour: Grey. Dimensions (WxDxH): 650 x 270 x 25.4 mm, Weight: 2.12 kg",Computers & Electronics>Data Input Devices>Wrist Rests,Computers & Electronics,Data Input Devices,Wrist Rests,"3M Gel Platform wrist rest Grey | 3M | Gel Wrist Rest Platform | Wrist Rests | 3M Gel Wrist Rest Platform. Product colour: Grey. Dimensions (WxDxH): 650 x 270 x 25.4 mm, Weight: 2.12 kg | 3M Gel Wrist Rest Platform, Grey, 650 x 270 x 25.4 mm, 2.12 kg | Gel Wrist Rest Platform | Sleek in appearance and in operation, this platform is designed with tapered wrist rests and rounded corners for comfort beyond compare. Use the tilt feature to help keep wrists properly aligned and<br>\r\nimprove ergonomic posture.<br>\r\n<br>\r\n- Keyboard and mouse wrist rest with antimicrobial product protection adds comfort by minimizing pressure points<br>\r\n- Battery saving design on mouse surfaces extends battery life of wireless mice up to 75%†<br>\r\n- Leatherette surfaces on wrist rests are buttery-soft and easy to keep clean<br>\r\n<br>\r\n- Tilt-adjustable platform for keyboard and mouse<br>\r\n- Mousing surface is repositionable for left-handed mousing","3m gel platform wrist rest grey 3m gel wrist rest platform wrist rests 3m gel wrist rest platform. product colour grey. dimensions wxdxh 650 x 270 x 25.4 mm, weight 2.12 kg 3m gel wrist rest platform, grey, 650 x 270 x 25.4 mm, 2.12 kg gel wrist rest platform sleek in appearance and in operation, this platform is designed with tapered wrist rests and rounded corners for comfort beyond compare. use the tilt feature to help keep wrists properly aligned and improve ergonomic posture. - keyboard and mouse wrist rest with antimicrobial product protection adds comfort by minimizing pressure points - battery saving design on mouse surfaces extends battery life of wireless mice up to 75 - leatherette surfaces on wrist rests are buttery-soft and easy to keep clean - tilt-adjustable platform for keyboard and mouse - mousing surface is repositionable for left-handed mousing"
16343,Backshop,8001000,Armrest,NaN,NaN,Backshop Armrest. Product colour: Black. Dimensions (WxDxH): 190 x 220 x 20 mm,Computers & Electronics>Data Input Devices>Wrist Rests,Computers & Electronics,Data Input Devices,Wrist Rests,"Backshop Armrest wrist rest Black | Backshop | Armrest | Wrist Rests | Backshop Armrest. Product colour: Black. Dimensions (WxDxH): 190 x 220 x 20 mm | Backshop Armrest, Black, 190 x 220 x 20 mm | |","backshop armrest wrist rest black backshop armrest wrist rests backshop armrest. product colour black. dimensions wxdxh 190 x 220 x 20 mm backshop armrest, black, 190 x 220 x 20 mm"
16344,3M,7000081010,7000081010,NaN,NaN,3M 7000081010. Product colour: Black. Dimensions (WxDxH): 71 x 483 x 20 mm,Computers & Electronics>Data Input Devices>Wrist Rests,Computers & Electronics,Data Input Devices,Wrist Rests,"3M 7000081010 wrist rest Black | 3M | 7000081010 | Wrist Rests | 3M 7000081010. Product colour: Black. Dimensions (WxDxH): 71 x 483 x 20 mm | 3M 7000081010, Black, 71 x 483 

In [16]:
sample_df[sample_df["Level3"] == "Wireless Routers"].nunique

<bound method DataFrame.nunique of                 Brand         BrandPartCode  \
16164           Zyxel         KEENETIC VIVA   
16165         Linksys            WAG320N-EU   
16166            ASUS       90IG03W1-BA1000   
16167         Netgear         MR1100-100EUS   
16168         Netgear          RBS50-100PES   
16169          D-Link               DWR-953   
16170        LevelOne             WBR-3470B   
16171            ASUS       90IG02P1-BO3110   
16172         Digicom                8E4566   
16173         Netgear        WNDRMAC-100PES   
16174  Allied Telesis         AT-WR2304N-10   
16175         Linksys             EA3500-EW   
16176           Zyxel               NBG5715   
16177            ASUS              RP-AC68U   
16178            ASUS       90IG01F1-BM2G00   
16179           Zyxel        91-004-594001B   
16180             AVM              20002778   
16181            ASUS    90-IG1G002M00-3PA0   
16182            ASUS       90IG0201-BU9G00   
16183         Digicom    

In [28]:
sample_df[sample_df["Level3"] == "Kids' Tablet Accessories"]

,Brand,BrandPartCode,ProductName,Description.LongProductName,Description.LongDesc,SummaryDescription.LongSummaryDescription,pathlist_names,Level1,Level2,Level3,metadata_text,metadata_clean
5997,VTech,80-230423,Game Mickey Mouse Clubhouse,NaN,NaN,"VTech Game Mickey Mouse Clubhouse, Storio. Recommended age (min): 4 yr(s), Recommended gender: Boy/Girl, Recommended age (max): 7 yr(s). Weight: 80 g. Package width: 20 mm, Package depth: 138 mm, Package height: 132 mm",Computers & Electronics>Computers>Kids' Tablet Accessories,Computers & Electronics,Computers,Kids' Tablet Accessories,"VTech Storio Game Mickey Mouse Clubhouse | VTech | Game Mickey Mouse Clubhouse | Kids' Tablet Accessories | VTech Game Mickey Mouse Clubhouse, Storio. Recommended age (min): 4 yr(s), Recommended gender: Boy/Girl, Recommended age (max): 7 yr(s). Weight: 80 g. Package width: 20 mm, Package depth: 138 mm, Package height: 132 mm | VTech Storio Game Mickey Mouse Clubhouse, 4 yr(s), Boy/Girl, 7 yr(s), Dutch, Multicolor, CE | |","vtech storio game mickey mouse clubhouse vtech game mickey mouse clubhouse kids tablet accessories vtech game mickey mouse clubhouse, storio. recommended age min 4 yr s , recommended gender boy/girl, recommended age max 7 yr s . weight 80 g. package width 20 mm, package depth 138 mm, package height 132 mm vtech storio game mickey mouse clubhouse, 4 yr s , boy/girl, 7 yr s , dutch, multicolor, ce"
5998,VTech,80-234004,80-234004,NaN,NaN,"VTech 80-234004. Language version: German, Distribution type: Physical media, Media type: Memory card",Computers & Electronics>Computers>Kids' Tablet Accessories,Computers & Electronics,Computers,Kids' Tablet Accessories,"VTech 80-234004 kids' tablet accessory | VTech | 80-234004 | Kids' Tablet Accessories | VTech 80-234004. Language version: German, Distribution type: Physical media, Media type: Memory card | VTech 80-234004, German, Physical media, Memory card | |","vtech 80-234004 kids tablet accessory vtech 80-234004 kids tablet accessories vtech 80-234004. language version german, distribution type physical media, media type memory card vtech 80-234004, german, physical media, memory card"
5999,VTech,80-274504,80-274504,NaN,NaN,"VTech 80-274504. Language version: German, Distribution type: Physical media, Media type: Memory card",Computers & Electronics>Computers>Kids' Tablet Accessories,Computers & Electronics,Computers,Kids' Tablet Accessories,"VTech 80-274504 kids' tablet accessory | VTech | 80-274504 | Kids' Tablet Accessories | VTech 80-274504. Language version: German, Distribution type: Physical media, Media type: Memory card | VTech 80-274504, German, Physical media, Memory card | |","vtech 80-274504 kids tablet accessory vtech 80-274504 kids tablet accessories vtech 80-274504. language version german, distribution type physical media, media type memory card vtech 80-274504, german, physical media, memory card"
6000,VTech,80-231805,Planes,NaN,NaN,"VTech Planes, Jeu Storio. Recommended age (min): 4 yr(s), Recommended age (max): 7 yr(s). Package width: 20 mm, Package depth: 140 mm, Package height: 155 mm",Computers & Electronics>Computers>Kids' Tablet Accessories,Computers & Electronics,Computers,Kids' Tablet Accessories,"VTech Jeu Storio Planes | VTech | Planes | Kids' Tablet Accessories | VTech Planes, Jeu Storio. Recommended age (min): 4 yr(s), Recommended age (max): 7 yr(s). Package width: 20 mm, Package depth: 140 mm, Package height: 155 mm | VTech Jeu Storio Planes, 4 yr(s), 7 yr(s), 20 mm, 140 mm, 155 mm, 120 g | |","vtech jeu storio planes vtech planes kids tablet accessories vtech planes, jeu storio. recommended age min 4 yr s , recommended age max 7 yr s . package width 20 mm, package depth 140 mm, package height 155 mm vtech jeu storio planes, 4 yr s , 7 yr s , 20 mm, 140 mm, 155 mm, 120 g"
6001,VTech,80-002181,Adapter duo 2.0,NaN,NaN,"VTech Adapter duo 2.0. Language version: Dutch, Material: Plastic, Product colour: Black. Width: 69 mm, Depth: 54 mm, Height: 84 mm",Computers & El

In [22]:
sample_df[sample_df["Level3"] == "Notebooks"].nunique()

Brand                                         11
BrandPartCode                                100
ProductName                                  100
Description.LongProductName                   81
Description.LongDesc                          78
SummaryDescription.LongSummaryDescription    100
pathlist_names                                 1
Level1                                         1
Level2                                         1
Level3                                         1
metadata_text                                100
metadata_clean                               100
dtype: int64

In [23]:
sample_df["Level3"].unique()

array(['AV Conferencing Bridges', 'AV Extenders', 'AV Receivers',
       'Activity Trackers', 'All-in-One PC/Workstation Mounts & Stands',
       'All-in-One PCs/Workstations', 'Antivirus Security Software',
       'Audio Cables', 'Audio Turntables', 'Backpack PCs',
       'Battery Chargers', 'Bridges & Repeaters', 'CD Players',
       'CPU Holders', 'CRT TVs', 'Cable Boots',
       'Cable Interface/Gender Adapters', 'Cable Protectors',
       'Cable Splitters or Combiners', 'Calculators', 'Car Kits',
       'Cassette Players', 'Coaxial Cables',
       'Component (YPbPr) Video Cables', 'Composite Video Cables',
       'Computer Monitors', 'Computer TV Tuners', 'Customer Displays',
       'DVD/Blu-Ray Players', 'DVI Cables', 'Data Projectors',
       'Dictaphones', 'Digital Media Players', 'Digital Photo Frames',
       'Display Privacy Filters', 'DisplayPort Cables',
       'Docking Speakers', 'E-Book Reader Cases', 'E-Book Readers',
       'Embedded Computers', 'Fax Machines', 'Fax Su

In [24]:
sample_df["Level3"].nunique()

209

In [35]:
sample_df["Level2"].nunique()

14